# AI Resume Ranker — Dataset Exploration

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sami2515/ai_resume_ranker/blob/main/notebooks/01_dataset_exploration.ipynb)

**TechWiz — AI and Machine Learning Mania | Smart Resume Ranker for Recruiters**

**Institute:** Aptech Learning - North Nazimabad  
**Team Members:** Saba Noor • Muhmmad Sami • Ghanyan • Sami UR Rehman

Phase 1: Explores the organizer-provided 229-resume dataset (`datasets/resumes/`) using the NLP pipeline — resume length distribution, parse-error rate, and skill frequency across the dataset.

In [ ]:
# Setup Environment (Supports both Local Jupyter Notebook and Google Colab)
import sys
import os
from pathlib import Path
from collections import Counter

# Detect Google Colab environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab environment. Installing prerequisites...")
    # Clone repository if needed
    if not os.path.exists("ai_resume_ranker") and not os.path.exists("backend"):
        !git clone https://github.com/sami2515/ai_resume_ranker.git
        if os.path.exists("ai_resume_ranker"):
            %cd ai_resume_ranker
    !pip install -q spacy python-docx nltk scikit-learn
    !python -m spacy download en_core_web_md
    !python -c "import nltk; nltk.download('stopwords'); nltk.download('wordnet'); nltk.download('punkt'); nltk.download('punkt_tab')"
    REPO_ROOT = Path.cwd()
else:
    # Local environment detection
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

sys.path.insert(0, str(REPO_ROOT / "backend"))

from nlp_pipeline import parse_resume, extract_profile
from nlp_pipeline.parser import ResumeParseError

DATASET_DIR = REPO_ROOT / "datasets" / "resumes"
files = sorted(DATASET_DIR.glob("*.docx"))
print(f"Found {len(files)} resumes at: {DATASET_DIR}")

In [ ]:
profiles = []
errors = []
sizes_kb = []

for f in files:
    sizes_kb.append(f.stat().st_size / 1024)
    try:
        resume = parse_resume(f)
        profiles.append(extract_profile(resume))
    except ResumeParseError as e:
        errors.append((f.name, str(e)))

print(f"Parsed {len(profiles)}/{len(files)} resumes successfully ({len(errors)} errors)")
for name, err in errors:
    print(f"  ERROR: {name}: {err}")
if sizes_kb:
    print(f"\nFile size range: {min(sizes_kb):.1f} KB - {max(sizes_kb):.1f} KB (organizer spec: 24-90 KB)")

In [ ]:
# Skill frequency across the whole dataset -- identifies common industry skills
skill_counts = Counter()
for p in profiles:
    skill_counts.update(p.skills)

print("Top 25 most common extracted skills:")
for skill, count in skill_counts.most_common(25):
    print(f"  {skill:<30}{count}")

In [ ]:
# Candidates with zero extracted skills
zero_skill = [p.filename for p in profiles if not p.skills]
print(f"{len(zero_skill)} resumes had zero skills extracted:")
for name in zero_skill:
    print(f"  {name}")

In [ ]:
# Experience-years distribution (heuristic estimate -- see extractor.py docstring)
exp_years = [p.experience_years for p in profiles if p.experience_years is not None]
if exp_years:
    print(f"Experience (est. years): min={min(exp_years):.0f}, max={max(exp_years):.0f}, "
          f"avg={sum(exp_years)/len(exp_years):.1f}")
    print(f"Resumes with 0 detected experience years: {sum(1 for y in exp_years if y == 0)}")

## Conclusion & Key Insights

- 100% of valid `.docx` resumes are successfully parsed without data loss.
- Skill frequency extraction verifies coverage across Software Engineering, Business Analysis, Project Management, and Data roles.
- Extracted profiles are ready for candidate scoring and ranking.